In [0]:
from pyspark.sql import functions as F

base_path = "/Workspace/Users/durga20010615@gmail.com/healthcare-data-engineering-pipeline/data/raw"

providers_df = spark.read.option("header", True).csv(f"{base_path}/providers.csv")
members_df = spark.read.option("header", True).csv(f"{base_path}/members.csv")


print("Providers:", providers_df.count())
print("Members:", members_df.count())

In [0]:
from pathlib import Path

incoming_dir = Path(
    "/Workspace/Users/durga20010615@gmail.com/healthcare-data-engineering-pipeline/data/incoming"
)

incoming_files = sorted(incoming_dir.glob("claims_batch_*.csv"))

print("Incoming files:", [f.name for f in incoming_files])


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS processed_claim_files (
    file_name STRING,
    processed_at TIMESTAMP
)
USING DELTA
""")

In [0]:
processed_files = spark.table("processed_claim_files")

processed_names = [
    row["file_name"] for row in processed_files.select("file_name").collect()
]

unprocessed_files = [
    file for file in incoming_files
    if file.name not in processed_names
]

print("Files to process:", [file.name for file in unprocessed_files])

if unprocessed_files:
    claims_df = spark.read.option("header", True).csv(
        [str(file) for file in unprocessed_files]
    )
    print("Incoming claims:", claims_df.count())
else:
    claims_df = None
    print("No new files to process")


In [0]:
providers_df.printSchema()
members_df.printSchema()

if claims_df is not None:
    claims_df.printSchema()

In [0]:
providers_df.write.mode("overwrite").format("delta").saveAsTable("bronze_providers")
members_df.write.mode("overwrite").format("delta").saveAsTable("bronze_members")

if claims_df is not None:
    existing_claims = spark.table("bronze_claims").select("claim_id")

    new_claims_df = claims_df.join(
        existing_claims,
        on="claim_id",
        how="left_anti"
    )

    new_claims_df.write.mode("append").format("delta").saveAsTable("bronze_claims")

    print("New claims added:", new_claims_df.count())
else:
    print("No new claims to add")

print("Bronze table data created successfully")


In [0]:
if claims_df is not None:
    processed_rows = [
        (file.name,)
        for file in unprocessed_files
    ]

    processed_df = (
        spark.createDataFrame(processed_rows, ["file_name"])
        .withColumn("processed_at", F.current_timestamp())
    )

    processed_df.write.mode("append").format("delta").saveAsTable(
        "processed_claim_files"
    )

    print("Files marked processed:", [file.name for file in unprocessed_files])


In [0]:
spark.sql("SHOW TABLES").show()

In [0]:
display(spark.table("bronze_providers"))

In [0]:
display(spark.table("bronze_members"))

In [0]:
display(spark.table("bronze_claims"))

In [0]:
display(spark.table("processed_claim_files"))